# 🎯 6. Re-ranking & Hybrid Search

Σε αυτό το notebook θα δούμε δύο τεχνικές που **σχεδόν πάντα** ανεβάζουν την ποιότητα ενός RAG συστήματος:

1. **Hybrid Search** — συνδυασμός sparse (lexical, BM25) + dense (semantic, embeddings)
2. **Re-ranking** — ένα δεύτερο, πιο ακριβές μοντέλο που αξιολογεί τα top αποτελέσματα

Η συνηθισμένη pipeline production ενός σοβαρού RAG είναι:

<img src="images/rerank-and-hybrid-search.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

<img src="images/two_stage_retrieval.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 6.1** — Two-Stage Retrieval: Stage 1 (Hybrid recall, fast, k=20) → Stage 2 (Reranker precision, k=3) → LLM.*

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
if not os.environ.get("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

## 6.1 Setup

In [3]:
# ── Imports ────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── LLM & Embeddings ───────────────────────────────────────────────────
llm        = ChatOpenAI(model=LLM_MODEL, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# ── Datanous.ai Knowledge Base ─────────────────────────────────────────
# Καθαρό corpus: 10 ρεαλιστικά product docs χωρίς "παγίδες".
# Τα tricky scenarios (decoys που μπερδεύουν το Dense) θα τα δούμε
# σε ξεχωριστό demo_kb στο 6.2.
datanous_kb = [
    Document(page_content="Datanous Insight Professional costs 350 USD per month and supports up to 100,000 document pages with multi-tenant row-level access control.", metadata={"product": "Insight Professional"}),
    Document(page_content="Datanous Insight Enterprise offers unlimited document pages, dedicated infrastructure, SOC 2 Type II certification, and GDPR compliance tools.", metadata={"product": "Insight Enterprise"}),
    Document(page_content="Datanous Insight Starter is priced at 99 USD per month and supports up to 10,000 pages. Ideal for small teams.", metadata={"product": "Insight Starter"}),
    Document(page_content="Datanous Guard redacts PII, enforces tenant isolation, and provides field-level encryption for sensitive deployments.", metadata={"product": "Guard"}),
    Document(page_content="Datanous Embed supports text-embedding-3-small and text-embedding-3-large models for semantic search across document corpora.", metadata={"product": "Embed"}),
    Document(page_content="The Guard XL enterprise security package is registered under SKU CMP-G-XL-008. The code is used internally for license activation, billing and support ticket routing.", metadata={"product": "Guard XL", "sku": "CMP-G-XL-008"}),
    Document(page_content="All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest for data security.", metadata={"product": "Platform"}),
    Document(page_content="Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weaviate as a preferred integration partner.", metadata={"product": "Platform"}),
    Document(page_content="A law firm reduced document search time from 2.5 hours to 8 minutes using Datanous Insight Enterprise.", metadata={"product": "Insight Enterprise"}),
    Document(page_content="Datanous.ai is headquartered in Athens, Greece, with offices in London and Amsterdam.", metadata={"product": "Company"}),
]
kb = datanous_kb

# ── Dense Retriever (Chroma in-memory, με fresh collection σε κάθε run) ─
import uuid
vectorstore     = Chroma.from_documents(
    datanous_kb,
    embedding=embeddings,
    collection_name=f"datanous_{uuid.uuid4().hex[:8]}",   # αποφεύγει double-indexing
)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# ── Test Queries ────────────────────────────────────────────────────────
# specific_q — αναφέρεται σε συγκεκριμένο SKU (ID-heavy)
# semantic_q — εννοιολογικό ερώτημα χωρίς keyword overlap με το σωστό doc
specific_q = "What is the SKU CMP-G-XL-008?"
semantic_q = "Which product protects against data leakage in multi-tenant environments?"

print("Setup complete.")
print(f"   KB documents : {len(datanous_kb)}")
print(f"   Specific query: {specific_q}")
print(f"   Semantic query: {semantic_q}")

Setup complete.
   KB documents : 10
   Specific query: What is the SKU CMP-G-XL-008?
   Semantic query: Which product protects against data leakage in multi-tenant environments?


In [4]:
# Dense retrieval results for both queries (for comparison with BM25 below)
print("Dense retrieval results:")
for q in [specific_q, semantic_q]:
    print(f"\nQuery: {q}")
    for d in dense_retriever.invoke(q)[:3]:
        print(f"  [{d.metadata.get('product','?')}] {d.page_content[:80]}")


Dense retrieval results:

Query: What is the SKU CMP-G-XL-008?
  [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G-XL-008. T
  [Embed] Datanous Embed supports text-embedding-3-small and text-embedding-3-large models
  [Guard] Datanous Guard redacts PII, enforces tenant isolation, and provides field-level 

Query: Which product protects against data leakage in multi-tenant environments?
  [Guard] Datanous Guard redacts PII, enforces tenant isolation, and provides field-level 
  [Insight Professional] Datanous Insight Professional costs 350 USD per month and supports up to 100,000
  [Insight Enterprise] Datanous Insight Enterprise offers unlimited document pages, dedicated infrastru


## 6.2 Όπου το dense search αποτυγχάνει

Οι embeddings είναι εξαιρετικές στο semantic match, αλλά **αδύναμες** σε:

* **Συγκεκριμένα IDs** (πχ `ORB-PUL-001`, `INC-2024-0431`, `ERR-9X4-K2887-A2`)
* **Σπάνιες, ακριβείς λέξεις** (πχ `MACsec`, `RPO`, `cl100k`)
* **Acronyms** (πχ `KMS`, `CMK`, `MFA`)

Ο λόγος: ένα alphanumeric ID όπως `ERR-9X4-K2887-A2` σπάει σε «νεκρά» subword tokens
που το embedding model δεν έχει δει ποτέ. Αντίθετα, το **BM25** (sparse, lexical)
το πιάνει ως ένα μοναδικό token με τεράστιο IDF score.

Ας το δείξουμε με ένα ελεγχόμενο πείραμα: φτιάχνουμε ένα μικρό KB όπου
**ένα μόνο doc** περιέχει το exact ID, και **8 decoys** παραφράζουν το ερώτημα
χωρίς όμως να περιέχουν το ID.

In [5]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Demo: Dense αποτυγχάνει, BM25 πετυχαίνει σε ID-heavy query         ║
# ╚══════════════════════════════════════════════════════════════════════╝
import uuid
from langchain_community.retrievers import BM25Retriever

# ── Purpose-built KB για αυτό το demo ──────────────────────────────────
# 1 σωστό doc (changelog-style, semantically άσχετο με το query)
# + 8 decoys που παραφράζουν το ερώτημα αλλά ΔΕΝ έχουν το ID
demo_kb = [
    # Σωστό doc — έχει το ID, αλλά «βαρετό» changelog context γύρω-γύρω
    Document(
        page_content=(
            "Datanous platform changelog — release 2.14.7 (September 2024). "
            "Nightly build pipeline patched: ERR-9X4-K2887-A2 resolved in staging. "
            "Deployment notes shipped to ops channel."
        ),
        metadata={"source": "changelog-2.14.7", "id": "ERR-9X4-K2887-A2"},
    ),
    # Decoys — όλα παραφράζουν το query, κανένα δεν έχει το ID
    Document(page_content="Looking up an error code in the Datanous platform is straightforward — every error has a unique identifier that you can reference.", metadata={"source": "docs-1"}),
    Document(page_content="Each error code in our system follows a standardized format. To find an error code, consult the error code reference manual.", metadata={"source": "docs-2"}),
    Document(page_content="Error codes help support teams diagnose issues quickly. When you encounter one, search the error code catalog for its meaning.", metadata={"source": "docs-3"}),
    Document(page_content="The complete error code reference is published in our developer documentation. Every error code is documented with a description and resolution.", metadata={"source": "docs-4"}),
    Document(page_content="Customers frequently ask where to find documentation for a specific error code. Always check the official error code index first.", metadata={"source": "docs-5"}),
    Document(page_content="When troubleshooting, locate the error code in the logs, then look up its documented meaning in the error reference guide.", metadata={"source": "docs-6"}),
    Document(page_content="Our error code system covers infrastructure, billing, security and runtime issues. Each error code is uniquely documented.", metadata={"source": "docs-7"}),
    Document(page_content="Datanous support requires the error code from your logs before opening a ticket. The error code helps route the issue to the right team.", metadata={"source": "docs-8"}),
]

# ── Retrievers (fresh collection για να μη διπλο-indexάρει σε re-runs) ─
demo_vs    = Chroma.from_documents(
    demo_kb,
    embedding=embeddings,
    collection_name=f"demo_fail_{uuid.uuid4().hex[:8]}",
)
demo_dense = demo_vs.as_retriever(search_kwargs={"k": 3})
demo_bm25  = BM25Retriever.from_documents(demo_kb); demo_bm25.k = 3

# ── Test query: περιέχει το exact ID ───────────────────────────────────
demo_q = "Where is error code ERR-9X4-K2887-A2 documented?"
print(f"Query: {demo_q}\n")

print("DENSE retrieval (top-3) — αναμένουμε αποτυχία:")
for i, d in enumerate(demo_dense.invoke(demo_q), 1):
    hit = "[ΟΚ]" if "ERR-9X4-K2887-A2" in d.page_content else "[Χ]"
    print(f"  {i}. {hit} [{d.metadata['source']:18s}] {d.page_content[:70]}…")

print("\n[ΟΚ] BM25 retrieval (top-3) — αναμένουμε επιτυχία:")
for i, d in enumerate(demo_bm25.invoke(demo_q), 1):
    hit = "[ΟΚ]" if "ERR-9X4-K2887-A2" in d.page_content else "[Χ]"
    print(f"  {i}. {hit} [{d.metadata['source']:18s}] {d.page_content[:70]}…")

Query: Where is error code ERR-9X4-K2887-A2 documented?

DENSE retrieval (top-3) — αναμένουμε αποτυχία:
  1. [Χ] [docs-6            ] When troubleshooting, locate the error code in the logs, then look up …
  2. [Χ] [docs-7            ] Our error code system covers infrastructure, billing, security and run…
  3. [Χ] [docs-2            ] Each error code in our system follows a standardized format. To find a…

[ΟΚ] BM25 retrieval (top-3) — αναμένουμε επιτυχία:
  1. [Χ] [docs-4            ] The complete error code reference is published in our developer docume…
  2. [Χ] [docs-7            ] Our error code system covers infrastructure, billing, security and run…
  3. [ΟΚ] [changelog-2.14.7  ] Datanous platform changelog — release 2.14.7 (September 2024). Nightly…


In [6]:
# !pip install -q rank-bm25
from langchain_community.retrievers import BM25Retriever

bm25   = BM25Retriever.from_documents(datanous_kb)
bm25.k = 4

print("BM25 (sparse) retrieval results:")
for q in [specific_q, semantic_q]:
    print(f"\nQuery: {q}")
    for d in bm25.invoke(q):
        print(f"  [{d.metadata.get('product','?')}] {d.page_content[:80]}")


BM25 (sparse) retrieval results:

Query: What is the SKU CMP-G-XL-008?
  [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G-XL-008. T
  [Company] Datanous.ai is headquartered in Athens, Greece, with offices in London and Amste
  [Platform] Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weaviate as a prefe
  [Insight Starter] Datanous Insight Starter is priced at 99 USD per month and supports up to 10,000

Query: Which product protects against data leakage in multi-tenant environments?
  [Platform] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest for data 
  [Company] Datanous.ai is headquartered in Athens, Greece, with offices in London and Amste
  [Insight Professional] Datanous Insight Professional costs 350 USD per month and supports up to 100,000
  [Insight Enterprise] A law firm reduced document search time from 2.5 hours to 8 minutes using Datano


**Τι βλέπουμε:**

* Το Dense δίνει στις πρώτες θέσεις docs που λένε «κάπου υπάρχει το manual των error codes»
  — σωστή σημασιολογία, **λάθος απάντηση**.
* Το BM25 πιάνει το exact ID-match. Για queries με συγκεκριμένα IDs, codes, ή σπάνιες
  λέξεις, το BM25 είναι **απαραίτητο**.

Άρα: χρειαζόμαστε **και τα δύο**. Αυτό ακριβώς θα κάνει το Hybrid Search στο 6.5.

Συχνά το dense search θα σου επιστρέψει σχετικά μεν, **αλλά όχι το ακριβές match** του SKU.

## 6.3 BM25 retriever — δυνατό σε keywords, αδύναμο σε σημασία

Το **BM25** (Best Match 25) είναι ένα κλασικό sparse algorithm:
σπάνιες λέξεις σε όλο το corpus αλλά συχνές στο doc → υψηλό score.

Στο 6.2 είδαμε ότι το BM25 **σώζει** το Dense όταν το query έχει
συγκεκριμένο ID. Αλλά τι γίνεται στο αντίθετο σενάριο — όταν το query
είναι **εννοιολογικό** και δεν μοιράζεται keywords με το σωστό doc;

Ας δοκιμάσουμε και τα δύο queries στο κανονικό μας KB (`datanous_kb`).

In [7]:
# ── BM25 πάνω στο main datanous_kb ─────────────────────────────────────
# !pip install -q rank-bm25
from langchain_community.retrievers import BM25Retriever

bm25   = BM25Retriever.from_documents(datanous_kb)
bm25.k = 4

print("BM25 (sparse) retrieval results:\n")
for q in [specific_q, semantic_q]:
    print(f"Query: {q}")
    for d in bm25.invoke(q):
        print(f"  [{d.metadata.get('product','?')}] {d.page_content[:80]}")
    print()

BM25 (sparse) retrieval results:

Query: What is the SKU CMP-G-XL-008?
  [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G-XL-008. T
  [Company] Datanous.ai is headquartered in Athens, Greece, with offices in London and Amste
  [Platform] Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weaviate as a prefe
  [Insight Starter] Datanous Insight Starter is priced at 99 USD per month and supports up to 10,000

Query: Which product protects against data leakage in multi-tenant environments?
  [Platform] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest for data 
  [Company] Datanous.ai is headquartered in Athens, Greece, with offices in London and Amste
  [Insight Professional] Datanous Insight Professional costs 350 USD per month and supports up to 100,000
  [Insight Enterprise] A law firm reduced document search time from 2.5 hours to 8 minutes using Datano



BM25 (sparse) retrieval results:

Query: What is the SKU CMP-G-XL-008?
  
- [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G…
- [<κάτι άλλο που έχει "SKU" ή "what" ή "is">]
- ...

Query: Which product protects against data leakage in multi-tenant environments?
  
- [<κάτι με "multi-tenant" ή "product">]   ← όχι απαραίτητα Guard
- ...

Εδώ βλέπουμε την επιτυχία του `BM25`:

Στο specific query (ID/SKU) βάζει το σωστό **Guard XL** πρώτο, γιατί πιάνει το exact keyword CMP-G-XL-008 που υπάρχει μόνο σε αυτό το document.

**Παρατήρηση:**

* **Specific query (`CMP-G-XL-008`)** → το BM25 βάζει το **Guard XL** πρώτο,
  γιατί πιάνει το exact keyword που υπάρχει **μόνο** σε αυτό το document. 
* **Semantic query (data leakage / multi-tenant)** → η σωστή απάντηση (`Datanous Guard`)
  μπορεί να **μην** εμφανιστεί ψηλά, γιατί δεν μοιράζεται keywords με το query.
  Το Guard doc λέει "redacts PII, enforces tenant isolation" — το query λέει
  "protects against data leakage". Σημασιολογικά ίδιο, λεξιλογικά εντελώς διαφορετικό. 

Ας το δούμε καθαρά:

In [9]:
# Δείχνουμε ξεκάθαρα τη δυνατότητα και την αδυναμία του BM25
# πάνω στις δύο test queries.

print("─" * 70)
print("BM25 στο SPECIFIC query (keyword/ID match — strong):")
print(f"  Q: {specific_q}")
top = bm25.invoke(specific_q)[0]
print(f"  → [{top.metadata.get('product','?')}] {top.page_content[:80]}...")
print()

print("─" * 70)
print("BM25 στο SEMANTIC query (no keyword overlap — weak):")
print(f"  Q: {semantic_q}")
for i, d in enumerate(bm25.invoke(semantic_q)[:3], 1):
    marker = "[ΟΚ]" if d.metadata.get("product") == "Guard" else "[Χ]"
    print(f"  {i}. {marker} [{d.metadata.get('product','?')}] {d.page_content[:80]}...")

print()
print("Notice: στο semantic query, η σχετική 'Datanous Guard' σελίδα")
print(" - συχνά ΔΕΝ έρχεται πρώτη — δεν μοιράζεται keywords με το query.")
print(" - Χρειαζόμαστε dense retrieval για να πιάσουμε τη σημασία.")

──────────────────────────────────────────────────────────────────────
BM25 στο SPECIFIC query (keyword/ID match — strong):
  Q: What is the SKU CMP-G-XL-008?
  → [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G-XL-008. T...

──────────────────────────────────────────────────────────────────────
BM25 στο SEMANTIC query (no keyword overlap — weak):
  Q: Which product protects against data leakage in multi-tenant environments?
  1. [Χ] [Platform] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest for data ...
  2. [Χ] [Company] Datanous.ai is headquartered in Athens, Greece, with offices in London and Amste...
  3. [Χ] [Insight Professional] Datanous Insight Professional costs 350 USD per month and supports up to 100,000...

Notice: στο semantic query, η σχετική 'Datanous Guard' σελίδα
 - συχνά ΔΕΝ έρχεται πρώτη — δεν μοιράζεται keywords με το query.
 - Χρειαζόμαστε dense retrieval για να πιάσουμε τη σημασία.


──────────────────────────────────────────────────────────────────────
BM25 στο SPECIFIC query (keyword/ID match — strong):
- Q: What is the SKU CMP-G-XL-008?
- → [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G…

──────────────────────────────────────────────────────────────────────
BM25 στο SEMANTIC query (no keyword overlap — weak):
  Q: Which product protects against data leakage in multi-tenant environments?
  1. [Χ] [<πιθανότατα Insight Professional ή Platform>]
  2. [Χ] [<κάτι άλλο>]
  3. [Χ] [<κάτι άλλο>]   ← το Guard doc μπορεί να μην είναι καν εδώ

💡 Notice: στο semantic query, η σχετική 'Datanous Guard' σελίδα
   συχνά ΔΕΝ έρχεται πρώτη — δεν μοιράζεται keywords με το query.
   Χρειαζόμαστε dense retrieval για να πιάσουμε τη σημασία.

**Σύνοψη μέχρι εδώ:**

| Query type | BM25 | Dense |
|---|---|---|
| ID / SKU / acronym | [ΟΚ] Δουλεύει εξαιρετικά | [Χ] Παραλίγο, αλλά εύκολα μπερδεύεται από decoys |
| Semantic / conceptual | [Χ] Συχνά αποτυγχάνει | [ΟΚ] Δουλεύει καλά |

Καμία από τις δύο μεθόδους δεν είναι παντογνώστρια.
**Hybrid search = το καλύτερο και των δύο κόσμων** → 6.4.

**Συμπέρασμα:** Καμία από τις δύο μεθόδους δεν είναι παντογνώστρια. **Hybrid** = best of both worlds.

## 6.4 Reciprocal Rank Fusion (RRF)

Έχουμε δύο retrievers που τα πάνε καλά σε διαφορετικά πράγματα:

* **BM25** → δυνατό σε IDs, κωδικούς, σπάνιες λέξεις
* **Dense** → δυνατό σε σημασιολογικά queries

Πώς συνδυάζουμε τα δύο rankings σε ένα τελικό; Ένας απλός κίνδυνος είναι να
προσθέτουμε scores απευθείας — αλλά τα BM25 scores και τα cosine similarities
**δεν είναι στην ίδια κλίμακα**. Το BM25 μπορεί να δώσει 15.7, το cosine 0.83.

Η λύση: **Reciprocal Rank Fusion** — ξεχνάμε τα scores, κοιτάμε μόνο τις θέσεις
(ranks) και τις συνδυάζουμε με τον τύπο:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

όπου:
* `rank_r(d)` = η θέση του doc `d` στα αποτελέσματα του retriever `r` (ξεκινά από 1)
* `k` = σταθερά smoothing, παραδοσιακά **60** (από το αρχικό paper)

**Γιατί δουλεύει:**

* Ένα doc στη θέση **1** πιάνει score `1/61 ≈ 0.0164`
* Ένα doc στη θέση **10** πιάνει score `1/70 ≈ 0.0143`
* Ένα doc που εμφανίζεται **σε ΚΑΙ ΤΟΥΣ ΔΥΟ** retrievers αθροίζει — οπότε «κερδίζει»
  ακόμα κι αν δεν ήταν στο top-1 κανενός

Έτσι, ένα doc που είναι «σχετικά καλό» και στους δύο retrievers νικάει ένα doc
που είναι «τέλειο» στον έναν αλλά απών στον άλλο. Ακριβώς το robustness που θέλουμε.

> 📎 **Paper:** Cormack, Clarke & Buettcher (2009) — *Reciprocal Rank Fusion outperforms
> Condorcet and individual Rank Learning Methods.*

In [11]:
# ── Reciprocal Rank Fusion: συγχωνεύει rankings, αγνοεί τα raw scores ───
def reciprocal_rank_fusion(results_lists: list[list], k: int = 60) -> list[tuple]:
    """Merge multiple ranked lists with Reciprocal Rank Fusion.

    Args:
        results_lists: λίστα από ranked lists (κάθε μία από έναν retriever)
        k: smoothing constant — μεγάλο k μειώνει την επίδραση των top θέσεων

    Returns:
        Sorted list of (document, fused_score) tuples.
    """
    scores: dict[str, float]    = {}
    doc_map: dict[str, Document] = {}
    for results in results_lists:
        for rank, doc in enumerate(results):
            key = doc.page_content                # χρησιμοποιούμε content ως id
            doc_map[key] = doc
            scores[key]  = scores.get(key, 0) + 1 / (k + rank + 1)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_map[content], score) for content, score in ranked]


# ── Σύγκριση: BM25 alone vs Dense alone vs RRF fusion ──────────────────
for q in [specific_q, semantic_q]:
    bm25_results  = bm25.invoke(q)
    dense_results = dense_retriever.invoke(q)
    fused         = reciprocal_rank_fusion([bm25_results, dense_results])

    print(f"Query: {q}")
    print(f"  BM25  top-1  : [{bm25_results[0].metadata.get('product','?'):20s}] {bm25_results[0].page_content[:60]}...")
    print(f"  Dense top-1  : [{dense_results[0].metadata.get('product','?'):20s}] {dense_results[0].page_content[:60]}...")
    print(f"  RRF   top-1  : [{fused[0][0].metadata.get('product','?'):20s}] {fused[0][0].page_content[:60]}...")
    print(f"               (score={fused[0][1]:.4f})")
    print()

Query: What is the SKU CMP-G-XL-008?
  BM25  top-1  : [Guard XL            ] The Guard XL enterprise security package is registered under...
  Dense top-1  : [Guard XL            ] The Guard XL enterprise security package is registered under...
  RRF   top-1  : [Guard XL            ] The Guard XL enterprise security package is registered under...
               (score=0.0328)

Query: Which product protects against data leakage in multi-tenant environments?
  BM25  top-1  : [Platform            ] All Datanous.ai deployments use TLS 1.3 in transit and AES-2...
  Dense top-1  : [Guard               ] Datanous Guard redacts PII, enforces tenant isolation, and p...
  RRF   top-1  : [Platform            ] All Datanous.ai deployments use TLS 1.3 in transit and AES-2...
               (score=0.0320)



**Τι παρατηρούμε:**

* Στο **specific query**, και οι δύο retrievers συμφωνούν στο Guard XL doc.
  Το RRF το ενισχύει διπλά (παίρνει score από **και τους δύο**) και το κρατά #1.

* Στο **semantic query**, το BM25 χάνει την μπάλα — αλλά το Dense την πιάνει.
  Επειδή το Guard doc εμφανίζεται **έστω χαμηλά** και στους δύο, το RRF το ανεβάζει
  στο #1 του fused list.

Αυτό είναι το magic: **RRF δίνει robustness**. Δεν χρειάζεται ο ένας retriever
να είναι τέλειος — αρκεί ο συνδυασμός να συμφωνεί.

> 💡 **Variant με weights:** Στην προηγμένη εκδοχή, κάθε retriever έχει βάρος `w_r`:
> $$\text{RRF}(d) = \sum_{r} \frac{w_r}{k + \text{rank}_r(d)}$$
> Αυτό ακριβώς κάνει το `EnsembleRetriever` του LangChain, που θα δούμε αμέσως.

**Πρακτικά:** Συχνά τα βάρη ξεκινούν 50/50 και τα tunάρεις βάσει evaluation (notebook 08). Σε domains με πολλά IDs/codes (νομικό, ιατρικό, financial) το βάρος του BM25 ανεβαίνει.

Καλά νέα: δεν χρειάζεται να γράψεις το RRF χειροκίνητα κάθε φορά. Το LangChain το έχει έτοιμο μέσω του `EnsembleRetriever`.

## 6.5 Hybrid Search με `EnsembleRetriever`

Στο 6.4 γράψαμε το RRF χειροκίνητα. Σε production δεν χρειάζεται:
το LangChain παρέχει το **`EnsembleRetriever`** που:

* Τρέχει N retrievers **παράλληλα**
* Συγχωνεύει τα αποτελέσματα με **weighted RRF** (ίδιος τύπος όπως στο 6.4)
* Σου δίνει **βάρη** για να τονίσεις τον έναν retriever έναντι του άλλου

Είναι ακριβώς το ίδιο abstraction με αυτό που γράψαμε — απλά πιο καθαρό
και με ενσωματωμένο `weights` parameter.

In [12]:
from langchain_classic.retrievers import EnsembleRetriever

# ── Hybrid 50/50 — ισόρροπη αφετηρία ───────────────────────────────────
hybrid = EnsembleRetriever(
    retrievers=[bm25, dense_retriever],
    weights=[0.5, 0.5],
)

for q in [specific_q, semantic_q]:
    print(f"Query: {q}")
    for d in hybrid.invoke(q)[:4]:
        product = d.metadata.get("product", "?")
        print(f"  • [{product:22s}] {d.page_content[:70]}")
    print()

Query: What is the SKU CMP-G-XL-008?
  • [Guard XL              ] The Guard XL enterprise security package is registered under SKU CMP-G
  • [Company               ] Datanous.ai is headquartered in Athens, Greece, with offices in London
  • [Embed                 ] Datanous Embed supports text-embedding-3-small and text-embedding-3-la
  • [Platform              ] Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weaviate 

Query: Which product protects against data leakage in multi-tenant environments?
  • [Platform              ] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest
  • [Insight Professional  ] Datanous Insight Professional costs 350 USD per month and supports up 
  • [Guard                 ] Datanous Guard redacts PII, enforces tenant isolation, and provides fi
  • [Company               ] Datanous.ai is headquartered in Athens, Greece, with offices in London



**Παρατήρηση:** Το hybrid έλυσε το πρόβλημα ταυτόχρονα και για τις δύο queries —
χωρίς να ξέρει αν το query είναι ID-heavy ή semantic.

Αλλά τα **weights** δεν είναι πάντα 50/50. Σε domains με πολλά IDs (νομικό,
ιατρικό, financial, e-commerce SKUs) ανεβάζουμε το BM25 weight. Σε domains
με κυρίως φυσική γλώσσα (customer support chat, FAQ search) ανεβάζουμε το Dense.

Ας το δούμε στο specific query — μετακινούμε το βάρος από Dense-heavy σε BM25-heavy
και κοιτάμε αν αλλάζει το top-1:

In [13]:
# ── Επίδραση των weights στο top-1 του specific query ──────────────────
# Μικρό A/B: BM25-heavy vs Dense-heavy. Πλήρες grid → άσκηση 6.13
print(f"Query: {specific_q}\n")
for w_bm, w_de in [(0.2, 0.8), (0.8, 0.2)]:
    h = EnsembleRetriever(
        retrievers=[bm25, dense_retriever],
        weights=[w_bm, w_de],
    )
    top1 = h.invoke(specific_q)[0]
    label = "BM25-heavy " if w_bm > w_de else "Dense-heavy"
    print(f"  weights (BM25={w_bm}, Dense={w_de}) [{label}]:")
    print(f"    → [{top1.metadata.get('product','?')}] {top1.page_content[:70]}")
    print()

Query: What is the SKU CMP-G-XL-008?

  weights (BM25=0.2, Dense=0.8) [Dense-heavy]:
    → [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G

  weights (BM25=0.8, Dense=0.2) [BM25-heavy ]:
    → [Guard XL] The Guard XL enterprise security package is registered under SKU CMP-G



## 6.6 Γιατί χρειαζόμαστε reranking

Στο 6.5 είδαμε ότι το hybrid search **λύνει το πρόβλημα retrieval** —
βρίσκει σχετικά docs και για ID queries και για semantic queries.

Αλλά υπάρχει ένα δεύτερο πρόβλημα που το hybrid **δεν** το λύνει:
**η σειρά** μέσα στα top-k δεν είναι πάντα η σωστή.

Συχνά το πιο σχετικό doc βρίσκεται στις θέσεις 3-5, όχι στο #1.
Αν στείλεις τα top-3 στο LLM, μπορεί να τα συνδυάσει σωστά. Αλλά αν
στείλεις μόνο το top-1, ή αν χρησιμοποιείς long context όπου τα μεσαία
docs αγνοούνται (βλ. 6.11 — *Lost in the Middle*), θα έχεις λάθος απάντηση.

Ας το δούμε σε πράξη:

In [14]:
# ── Πότε το hybrid top-1 δεν είναι αρκετό ──────────────────────────────
# Το hybrid κάνει εξαιρετική δουλειά στο recall (βρίσκει το σωστό doc
# κάπου στα top-k). Αλλά το ranking δεν είναι πάντα τέλειο.

# Δοκιμάζουμε ένα query που έχει σχετικά πολλά υποψήφια docs:
multi_relevant_q = "What security and compliance features does Datanous.ai offer?"

print(f"Query: {multi_relevant_q}\n")
print("Hybrid recall (top-5):")
candidates = hybrid.invoke(multi_relevant_q)[:5]
for i, d in enumerate(candidates, 1):
    print(f"  {i}. [{d.metadata.get('product','?'):22s}] {d.page_content[:70]}")

Query: What security and compliance features does Datanous.ai offer?

Hybrid recall (top-5):
  1. [Insight Enterprise    ] Datanous Insight Enterprise offers unlimited document pages, dedicated
  2. [Platform              ] Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weaviate 
  3. [Guard XL              ] The Guard XL enterprise security package is registered under SKU CMP-G
  4. [Platform              ] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest
  5. [Company               ] Datanous.ai is headquartered in Athens, Greece, with offices in London


**Το πρόβλημα:** Το hybrid χρησιμοποιεί γρήγορα, χονδροειδή signals —
keyword overlap (BM25) και cosine similarity (Dense). Κανένα από τα δύο
**δεν διαβάζει το query και το doc μαζί** για να αποφασίσει σχετικότητα.

**Η λύση:** Δύο στάδια.

| Στάδιο | Μοντέλο | Ταχύτητα | Ακρίβεια | Έργο |
|---|---|---|---|---|
| **Stage 1 — Recall** | Bi-encoder (embeddings) + BM25 | ⚡⚡⚡ Πολύ γρήγορο | ⭐⭐ Μέτρια | Φέρνει k=20 υποψήφια από εκατομμύρια docs |
| **Stage 2 — Precision** | Cross-encoder ή LLM-as-judge | 🐢 Αργό | ⭐⭐⭐⭐ Πολύ καλή | Αναδιατάσσει τα 20 και κρατά τα top-3 |

**Γιατί δεν χρησιμοποιούμε τον reranker εξαρχής σε όλο το corpus;**

Ένας cross-encoder χρειάζεται ~50ms ανά (query, doc) pair. Με 1M docs αυτό
είναι **14 ώρες** ανά query. Με recall stage που φιλτράρει σε 20 candidates,
ο reranker χρειάζεται ~1 δευτερόλεπτο.

Άρα: **recall = γρήγορα + χονδρά, precision = αργά + ακριβή.**
Συνδυασμένα → καλύτερο cost/quality ratio.

Στα επόμενα 3 sections θα δούμε **τρεις τρόπους** να κάνουμε rerank:

* **6.7** — Cross-encoder local (γρήγορο, δωρεάν, καλή ποιότητα)
* **6.8** — Cohere Rerank (managed API, top quality, $)
* **6.9** — LLM-as-a-judge (πιο αργό αλλά customizable)

Και στο 6.10 θα τα ενώσουμε όλα σε ένα production pipeline.

## 6.7 Cross-Encoder reranker

Στους **bi-encoders** (αυτό που κάνει το embedding model):
query και doc περνάνε **ξεχωριστά** στο μοντέλο → ένα vector ο καθένας → cosine.
Γρήγορο, γιατί τα doc vectors είναι **προυπολογισμένα** offline.

Στους **cross-encoders**:
query και doc περνάνε **μαζί** ως ένα input στο Transformer → score απευθείας.
Πιο ακριβές, γιατί το μοντέλο «βλέπει» τις λεξιλογικές αλληλεπιδράσεις
μεταξύ query και doc. Αλλά αργό — πρέπει να ξανατρέξει για **κάθε** pair.

Bi-encoder:    Q → [Encoder] → v_q   ─┐
D → [Encoder] → v_d   ─┴→ cos(v_q, v_d)Cross-encoder: [Q, D] → [Transformer] → score

Για αυτό δουλεύει το **two-stage pipeline**: bi-encoder για να φιλτράρει
1M → 20 docs γρήγορα, cross-encoder για να ταξινομήσει αυτά τα 20 σωστά.

Εδώ θα χρησιμοποιήσουμε ένα ελαφρύ cross-encoder από Hugging Face:
**`cross-encoder/ms-marco-MiniLM-L-6-v2`** (~80MB, τρέχει σε CPU).

In [15]:
# Cross-Encoder reranker: query+doc μαζί σε Transformer
# !pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

# Lightweight cross-encoder fine-tuned σε MS MARCO (~80MB, CPU-friendly)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def cross_encoder_rerank(question: str, docs: list, top_k: int = 3):
    """Re-rank docs by feeding each (query, doc) pair to the cross-encoder.

    Returns sorted (doc, score) tuples. Higher score = more relevant.
    Note: το ms-marco model επιστρέφει logits, όχι probabilities —
    μπορεί να είναι αρνητικά. Σημασία έχει η σχετική σειρά.
    """
    pairs  = [(question, d.page_content) for d in docs]
    scores = cross_encoder.predict(pairs)
    return sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_k]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [16]:
# ── Two-stage retrieval: συνεχίζουμε το παράδειγμα του 6.6 ─────────────
# Στο 6.6 είδαμε ότι το hybrid βρίσκει σχετικά docs αλλά το ranking
# δεν είναι τέλειο. Ο cross-encoder το διορθώνει.

q = multi_relevant_q   # "What security and compliance features does Datanous.ai offer?"

# Stage 1 — γενναιόδωρο recall με hybrid
candidates = hybrid.invoke(q)

print(f"Query: {q}\n")
print(f"Stage 1 — Hybrid recall (top-{len(candidates)} candidates, αρχικό ranking):")
for i, d in enumerate(candidates, 1):
    print(f"  {i}. [{d.metadata.get('product','?'):22s}] {d.page_content[:65]}")

# Stage 2 — precision rerank
print("\nStage 2 — Cross-encoder rerank (top-3, με νέο score):")
for doc, score in cross_encoder_rerank(q, candidates, top_k=3):
    print(f"  [{score:+.3f}] [{doc.metadata.get('product','?'):22s}] {doc.page_content[:65]}")

Query: What security and compliance features does Datanous.ai offer?

Stage 1 — Hybrid recall (top-6 candidates, αρχικό ranking):
  1. [Insight Enterprise    ] Datanous Insight Enterprise offers unlimited document pages, dedi
  2. [Platform              ] Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weav
  3. [Guard XL              ] The Guard XL enterprise security package is registered under SKU 
  4. [Platform              ] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at
  5. [Company               ] Datanous.ai is headquartered in Athens, Greece, with offices in L
  6. [Guard                 ] Datanous Guard redacts PII, enforces tenant isolation, and provid

Stage 2 — Cross-encoder rerank (top-3, με νέο score):
  [+4.524] [Insight Enterprise    ] Datanous Insight Enterprise offers unlimited document pages, dedi
  [+3.636] [Platform              ] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at
  [+1.981] [Guard                 ]

**Πώς διαβάζονται τα scores:**

Το `ms-marco-MiniLM` επιστρέφει **logits**, όχι probabilities. Δηλαδή:

* Μπορούν να είναι **αρνητικά** (πχ `-4.5`) — αυτό δεν σημαίνει «αρνητικά σχετικό»
* Δεν είναι στο εύρος `[0, 1]`
* Έχει νόημα **μόνο η σχετική σειρά**: μεγαλύτερο score → πιο σχετικό για αυτό το query

Αν θες probabilities, μπορείς να βάλεις sigmoid:
```python
import numpy as np
prob = 1 / (1 + np.exp(-score))   # ή torch.sigmoid(score)
```
Αλλά για ranking δεν χρειάζεται — η σύγκριση raw scores δουλεύει.

**Cost analysis:**

| Pair | Latency (CPU) | Latency (GPU) |
|---|---|---|
| 1 (query, doc) | ~30ms | ~3ms |
| 20 candidates | ~600ms | ~60ms |
| 100 candidates | ~3s | ~300ms |

Για production-scale traffic, ο cross-encoder είναι ο **bottleneck**. Γι' αυτό
στο stage 1 κάνουμε **aggressive** filtering (κρατάμε 20-30 candidates, όχι 100+).

Ο local cross-encoder είναι **καλή αρχή**, αλλά:

* Χρειάζεται GPU για ταχύτητα production
* Δεν αναπροσαρμόζεται εύκολα σε domain-specific data χωρίς fine-tuning
* Η ποιότητα είναι ⭐⭐⭐ — όχι state-of-the-art

Αν θες managed quality χωρίς infrastructure, η **Cohere Rerank** δίνει
state-of-the-art αποτελέσματα μέσω API. → **6.8**

## 6.8 Cohere Rerank — managed service

Ο local cross-encoder είναι δωρεάν, αλλά έχει 3 περιορισμούς:

* **Ταχύτητα σε CPU**: ~30ms/pair → ~600ms για 20 candidates (αν δεν έχεις GPU)
* **Ποιότητα**: το `ms-marco-MiniLM` είναι μικρό μοντέλο — ⭐⭐⭐ από ⭐⭐⭐⭐⭐
* **Συντήρηση**: εσύ φροντίζεις το hosting, το versioning, τα GPU costs

Η **Cohere Rerank** προσφέρει managed reranking endpoint:

* **Ποιότητα ⭐⭐⭐⭐⭐** — fine-tuned σε hundreds of domains
* **Latency ~100-200ms** σε όλο το pipeline (έστω και με network round-trip)
* **Pricing $0.001/search** στο rerank-english-v3.0
* **Multilingual** support (rerank-multilingual-v3.0) — πιάνει και ελληνικά

Σε production είναι συχνά η **πρώτη επιλογή** για reranking. Free tier στο
[dashboard.cohere.com](https://dashboard.cohere.com/api-keys) (1000 calls/month
trial, αρκετό για development).

In [17]:
# Cohere Rerank — managed cross-encoder via API (production-grade)
# !pip install -q langchain-cohere cohere
# Requires: COHERE_API_KEY στο .env

import os

q = multi_relevant_q   # ίδιο query με 6.7 για apples-to-apples comparison

if os.environ.get("COHERE_API_KEY"):
    from langchain_cohere import CohereRerank
    from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain_classic.retrievers import EnsembleRetriever

    # ── Wrap το hybrid retriever με Cohere ως «compressor» ─────────────
    # Στη LangChain, ο reranker τρέχει σαν compressor: παίρνει τα candidates
    # του base_retriever και κρατάει μόνο τα top_n πιο σχετικά.
    compressor = CohereRerank(model="rerank-english-v3.0", top_n=3)
    cohere_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=hybrid,        # Stage 1: hybrid recall (από 6.5)
    )

    print(f"Query: {q}\n")
    print("Cohere-reranked (top-3):")
    for r in cohere_retriever.invoke(q):
        rel = r.metadata.get("relevance_score", float("nan"))
        product = r.metadata.get("product", "?")
        print(f"  [score={rel:.3f}] [{product:22s}] {r.page_content[:65]}")
else:
    print("⚠️  COHERE_API_KEY δεν βρέθηκε.")
    print("    Για να τρέξεις αυτό το section:")
    print("    1. Φτιάξε δωρεάν λογαριασμό: https://dashboard.cohere.com/api-keys")
    print("    2. Πρόσθεσε COHERE_API_KEY=... στο .env")
    print("    3. Restart kernel και ξανατρέξε αυτό το cell")
    print()
    print("    Εναλλακτικά, μπορείς να παραλείψεις αυτό το section —")
    print("    ο cross-encoder του 6.7 και το LLM-as-judge του 6.9 είναι λειτουργικές εναλλακτικές.")

⚠️  COHERE_API_KEY δεν βρέθηκε.
    Για να τρέξεις αυτό το section:
    1. Φτιάξε δωρεάν λογαριασμό: https://dashboard.cohere.com/api-keys
    2. Πρόσθεσε COHERE_API_KEY=... στο .env
    3. Restart kernel και ξανατρέξε αυτό το cell

    Εναλλακτικά, μπορείς να παραλείψεις αυτό το section —
    ο cross-encoder του 6.7 και το LLM-as-judge του 6.9 είναι λειτουργικές εναλλακτικές.


**Decision tree:**

* **Έχεις GPU + θες ελεγχόμενο cost** → cross-encoder local (6.7)
* **Production + budget για managed service** → **Cohere Rerank** (6.8)
* **Custom logic ή reasoning needed** (πχ «κράτα μόνο docs που αναφέρουν τιμή»)
  → LLM-as-judge (6.9)

Το LLM-as-judge είναι το πιο ευέλικτο — αλλά και το πιο ακριβό σε tokens.
Δες πότε αξίζει στο **6.9**.

## 6.9 LLM-as-a-Judge reranking

Αντί για dedicated reranking model (cross-encoder ή Cohere), μπορείς να ζητήσεις
από το **ίδιο το LLM** σου να βαθμολογήσει τη σχετικότητα κάθε candidate doc
προς το query.

**Γιατί να το κάνεις αυτό:**

* **Customizable prompt** — μπορείς να βάλεις domain rules:
  «δώσε υψηλότερο score αν το doc αναφέρει τιμή», «πέναλτι σε docs > 2 ετών»,
  «πρόσεξε για conflict of interest». Κανένας dedicated reranker δεν το προσφέρει αυτό.
* **Χωρίς extra dependencies** — έχεις ήδη το LLM client στο stack σου.
* **Reasoning δίπλα στο score** — μπορείς να ζητήσεις και εξήγηση («γιατί το score είναι 0.7;»).

**Γιατί να μην το κάνεις:**

* **Αργό**: 1 LLM call ανά candidate (~500ms-1s). Για 20 candidates → 10-20 δευτερόλεπτα sequential.
* **Ακριβό**: tokens για κάθε pair. Με `gpt-4o-mini` και 20 candidates ~$0.001-0.003 ανά query.
* **Λιγότερο calibrated** από Cohere — τα scores μπορεί να συγκεντρώνονται στο 0.8-1.0.

**Συμπέρασμα:** Καλή επιλογή για χαμηλό όγκο queries με ειδικές απαιτήσεις
ή για prototyping πριν επενδύσεις σε managed service.

In [ ]:
# LLM-as-Cross-Encoder: το LLM σου βαθμολογεί κάθε (query, doc) pair

from pydantic import BaseModel, Field

class RelevanceScore(BaseModel):
    """Relevance score for a retrieved document."""
    score: float = Field(description="Relevance score from 0.0 (not relevant) to 1.0 (highly relevant)")
    reason: str  = Field(description="Brief explanation of the score")

score_llm = llm.with_structured_output(RelevanceScore)

# Αυστηρή rubric — αλλιώς το LLM δίνει 1.00 παντού
RERANK_PROMPT = ChatPromptTemplate.from_template(
    """Score how well the document answers the question. Use this rubric:

  1.0  = Directly and fully answers the question
  0.7  = Partially answers — mentions key facts but lacks specifics
  0.4  = Same general topic but does not answer the specific question
  0.1  = Mentions related keywords only
  0.0  = Completely unrelated

Be strict. Most documents should score 0.4 or below — score 1.0 ONLY for direct answers.

Question: {question}
Document: {document}"""
)

def llm_rerank(question: str, docs: list, top_k: int = 3) -> list:
    """Re-rank documents using LLM relevance scores."""
    scored = []
    for doc in docs:
        result = (RERANK_PROMPT | score_llm).invoke(
            {"question": question, "document": doc.page_content}
        )
        scored.append((doc, result.score, result.reason))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

In [19]:
q = multi_relevant_q   # "What security and compliance features does Datanous.ai offer?"

# Stage 1 — hybrid recall (ίδιο pipeline όπως πριν)
candidates = hybrid.invoke(q)
print(f"Query: {q}\n")
print(f"Stage 1 — Hybrid recall ({len(candidates)} candidates)")

# Stage 2 — LLM-as-judge rerank
reranked = llm_rerank(q, candidates, top_k=3)

print(f"\nStage 2 — LLM-reranked (top-3, με reasoning):")
for doc, score, reason in reranked:
    product = doc.metadata.get("product", "?")
    print(f"  [{score:.2f}] [{product:22s}] {doc.page_content[:60]}")
    print(f"         reason: {reason[:90]}")

Query: What security and compliance features does Datanous.ai offer?

Stage 1 — Hybrid recall (6 candidates)

Stage 2 — LLM-reranked (top-3, με reasoning):
  [0.70] [Insight Enterprise    ] Datanous Insight Enterprise offers unlimited document pages,
         reason: The document mentions key security and compliance features such as SOC 2 Type II certifica
  [0.70] [Platform              ] All Datanous.ai deployments use TLS 1.3 in transit and AES-2
         reason: The document mentions key security features (TLS 1.3 and AES-256) but does not provide a c
  [0.70] [Guard                 ] Datanous Guard redacts PII, enforces tenant isolation, and p
         reason: The document partially answers the question by mentioning key security features like PII r


**Cost analysis (per query, 20 candidates, gpt-4o-mini):**

| Πτυχή | LLM-as-judge | Cross-encoder local | Cohere Rerank |
|---|---|---|---|
| Latency (sequential) | ~10-20s | ~600ms (CPU) | ~150ms |
| Latency (batched) | ~2-3s | ~600ms | ~150ms |
| Cost / query | ~$0.002 | $0 | $0.001 |
| Quality | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Customizable rules | ✅ | ❌ | ❌ |

**Speedup tip:** το LangChain LCEL επιτρέπει parallel calls με `.batch()`:

```python
# Αντί για sequential loop:
results = (RERANK_PROMPT | score_llm).batch([
    {"question": q, "document": d.page_content} for d in docs
])
```
Αυτό κατεβάζει τη latency από 10s σε ~2s για 20 candidates.

**Το πραγματικό plus του LLM-as-judge: customization.** Πχ αν θες να εφαρμόσεις
επιχειρηματικούς κανόνες, μπορείς να προσθέσεις στο prompt:

```python
RERANK_PROMPT_CUSTOM = ChatPromptTemplate.from_template(
    """[ίδια rubric...]

ADDITIONAL RULES:
- Bonus +0.2 αν το doc αναφέρει συγκεκριμένη τιμή
- Penalty -0.3 αν το doc είναι παλαιότερο από 2023
- Σκορ 0 αν το doc αναφέρει deprecated products

Question: {question}
Document: {document}"""
)
```
Αυτό είναι αδύνατο με έναν dedicated reranker χωρίς fine-tuning.

## 6.10 Full Production Pipeline

Ας ενώσουμε όλα όσα μάθαμε σε ένα production-grade RAG pipeline:

<img src="images/full_prod_pipeline.png" height="900px" width="700px" style="border-radius:10px;margin:12px 0;"/>

**Σχεδιαστικές επιλογές:**

* **Over-fetch στο stage 1** (k=8): δίνουμε στον reranker αρκετά υποψήφια. Σε production: k=20-30.
* **Aggressive filtering στο stage 2** (top_k=3): μειώνουμε τα docs στο context για να αποφύγουμε το *Lost in the Middle* (βλ. 6.11).
* **Grounded prompt**: ρητή οδηγία «Answer using ONLY the context» + «If unclear, say so».

Σε αυτό το demo χρησιμοποιούμε **LLM-as-judge** για το reranking γιατί είναι
αυτο-αναφερόμενο (χρησιμοποιεί το ίδιο LLM που είναι ήδη στο stack). Σε
πραγματικό production θα ήταν Cohere Rerank ή local cross-encoder για ταχύτητα.

In [20]:
# 🏭 Full Production Pipeline — χωρίς side effects, με local k settings

ANSWER_PROMPT = ChatPromptTemplate.from_template(
    """You are a Datanous.ai support assistant. Answer using ONLY the context below.
If the context doesn't contain the answer, say "I do not have that information."

Context:
{context}

Question: {question}

Answer:"""
)


def production_rag(question: str, k_recall: int = 8, k_final: int = 3) -> dict:
    """Three-stage RAG pipeline: hybrid recall → LLM rerank → grounded generation.

    Παράμετροι:
        question : το user query
        k_recall : πόσα candidates να φέρει το stage 1 (default 8, prod 20-30)
        k_final  : πόσα να κρατήσει ο reranker (default 3, σπάνια > 5)

    Returns:
        Dict με question, candidates, reranked, answer.

    Note: η function ορίζει τοπικά τον recall retriever ώστε να μη μεταλλάσσει
    το global state των bm25 και dense_retriever (καθαρά side effects).
    """
    # Stage 1 — local recall retriever με τοπικά k settings
    bm25_local = BM25Retriever.from_documents(datanous_kb); bm25_local.k = k_recall
    dense_local = vectorstore.as_retriever(search_kwargs={"k": k_recall})
    recall = EnsembleRetriever(
        retrievers=[bm25_local, dense_local],
        weights=[0.5, 0.5],
    )
    candidates = recall.invoke(question)[:k_recall]

    # Stage 2 — rerank (LLM-as-judge· swap με Cohere/cross-encoder σε prod)
    reranked = llm_rerank(question, candidates, top_k=k_final)
    top_docs = [doc for doc, _, _ in reranked]

    # Stage 3 — grounded generation
    context = "\n\n".join(f"- {d.page_content}" for d in top_docs)
    answer  = (ANSWER_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return {
        "question":   question,
        "candidates": candidates,
        "reranked":   reranked,
        "answer":     answer,
    }

In [21]:
# ── Demo με timing breakdown ───────────────────────────────────────────
import time

q = "What Datanous.ai product would you recommend for a bank that needs GDPR compliance?"

# Manual breakdown για να δούμε τι κοστίζει κάθε stage
t0 = time.perf_counter()

bm25_local = BM25Retriever.from_documents(datanous_kb); bm25_local.k = 8
dense_local = vectorstore.as_retriever(search_kwargs={"k": 8})
recall = EnsembleRetriever(retrievers=[bm25_local, dense_local], weights=[0.5, 0.5])
candidates = recall.invoke(q)[:8]
t1 = time.perf_counter()

reranked = llm_rerank(q, candidates, top_k=3)
top_docs = [doc for doc, _, _ in reranked]
t2 = time.perf_counter()

context = "\n\n".join(f"- {d.page_content}" for d in top_docs)
answer  = (ANSWER_PROMPT | llm | StrOutputParser()).invoke(
    {"context": context, "question": q}
)
t3 = time.perf_counter()

# ── Output ────────────────────────────────────────────────────────────
print(f"❓ Question:\n   {q}\n")

print(f"📥 Stage 1 — Hybrid recall ({len(candidates)} candidates) — {(t1-t0)*1000:.0f}ms")
for d in candidates:
    print(f"   • [{d.metadata.get('product','?'):22s}] {d.page_content[:65]}")

print(f"\n🎯 Stage 2 — LLM rerank (top-{len(reranked)}) — {(t2-t1)*1000:.0f}ms")
for doc, score, reason in reranked:
    print(f"   [{score:.2f}] [{doc.metadata.get('product','?'):22s}] {doc.page_content[:65]}")
    print(f"          reason: {reason[:75]}")

print(f"\n💬 Stage 3 — LLM Answer — {(t3-t2)*1000:.0f}ms")
print(f"   {answer}")

print(f"\n⏱️  Total: {(t3-t0)*1000:.0f}ms")
print(f"   Breakdown: recall={(t1-t0)*1000:.0f}ms · rerank={(t2-t1)*1000:.0f}ms · gen={(t3-t2)*1000:.0f}ms")

❓ Question:
   What Datanous.ai product would you recommend for a bank that needs GDPR compliance?

📥 Stage 1 — Hybrid recall (8 candidates) — 312ms
   • [Platform              ] Datanous.ai is certified by OpenAI, Anthropic, Pinecone, and Weav
   • [Insight Enterprise    ] Datanous Insight Enterprise offers unlimited document pages, dedi
   • [Company               ] Datanous.ai is headquartered in Athens, Greece, with offices in L
   • [Platform              ] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at
   • [Insight Enterprise    ] A law firm reduced document search time from 2.5 hours to 8 minut
   • [Guard                 ] Datanous Guard redacts PII, enforces tenant isolation, and provid
   • [Guard XL              ] The Guard XL enterprise security package is registered under SKU 
   • [Insight Professional  ] Datanous Insight Professional costs 350 USD per month and support

🎯 Stage 2 — LLM rerank (top-3) — 13257ms
   [1.00] [Insight Enterprise    ] Datano

**Πώς γίνεται production-ready αυτό το pipeline:**

| Αλλαγή | Latency impact | Effort |
|---|---|---|
| Swap LLM-as-judge → **Cohere Rerank** | -7.5s (10x speedup) | 5 γραμμές κώδικα |
| Swap LLM-as-judge → **cross-encoder GPU** | -7s | +GPU infra |
| Batch το LLM rerank | -6s | 1 γραμμή (χρήση `.batch()`) |
| Add **caching** (Redis) για επαναλαμβανόμενα queries | -all of it στο cache hit | Νέα υποδομή |
| Persistent **vectorstore** (Pinecone, Weaviate) αντί Chroma in-memory | Negligible read perf, +scalability | Νέα υποδομή |

**Code change για Cohere swap (το ίδιο pipeline, διαφορετικός reranker):**

```python
from langchain_cohere import CohereRerank
from langchain.retrievers import ContextualCompressionRetriever

compressor = CohereRerank(model="rerank-english-v3.0", top_n=3)
production_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=recall,
)

# Stage 2 γίνεται:
top_docs = production_retriever.invoke(question)
```

Δεν αλλάζει τίποτα στο stage 1 ή stage 3 — μόνο ο reranker.

Έχουμε ένα pipeline που δίνει σχετικά docs με σωστή σειρά. Μένει μια λεπτομέρεια:
**πόσα docs** δίνουμε τελικά στο LLM, και **σε ποια θέση**. Φαντάσου ότι περνάς
20 σχετικά docs στο context — θα τα διαβάσει όλα; Η έρευνα λέει **όχι**.

Δες το πρόβλημα *Lost in the Middle* → **6.11**.

***Εικ. 6.2** — Lost in the Middle: τα LLMs δίνουν λιγότερη προσοχή στα documents που βρίσκονται στη μέση του context window.*

## 6.11 Long Context Impact — «Lost in the Middle»

> **Part 18 του αρχικού RAG from Scratch** ([arxiv.org/abs/2307.03172](https://arxiv.org/pdf/2307.03172))

Φανταστείτε ότι ο retriever σας επέστρεψε 20 σχετικά documents και τα βάλατε
**όλα** στο context window. Ένα LLM με 128K context window μπορεί *τεχνικά*
να τα χωρέσει — αλλά τι γίνεται *πράγματι*;

### Το πρόβλημα

Η έρευνα δείχνει ότι τα LLMs **δεν δίνουν ίδια προσοχή** σε όλες τις θέσεις
μέσα στο context:

* ✅ **Αρχή** (positions 1–3): High attention
* ❌ **Μέση** (positions 4–17): Attention drops — το LLM συχνά **αγνοεί** αυτά τα docs
* ✅ **Τέλος** (positions 18–20): Moderate attention recovery

Αποτέλεσμα: ακόμα κι αν το πιο σχετικό doc βρίσκεται στο context, αν είναι
στη **μέση**, το LLM ενδέχεται να το παραβλέψει.

### Το πείραμα

Θα φτιάξουμε ένα context με **15 documents**, και θα τοποθετήσουμε το ένα
**σχετικό** doc σε 3 διαφορετικές θέσεις:

* Θέση 1 (αρχή) → περιμένουμε σωστή απάντηση
* Θέση 8 (μέση) → εδώ συχνά **αποτυγχάνει**
* Θέση 15 (τέλος) → συχνά σωστή λόγω recency bias

Ένας **LLM judge** θα αξιολογήσει κάθε απάντηση ως CORRECT ή INCORRECT.

> **Note:** Επεκτείνουμε το `datanous_kb` με πρόσθετα ψευδό-docs (Pipeline,
> Monitor, Trace, κ.ά.) για να φτάσουμε τα 15 — γιατί χρειάζεται **μεγάλο**
> context για να εμφανιστεί καθαρά το φαινόμενο. Τα παλιά μας 10 docs δεν αρκούν.

<img src="images/lost_in_the_middle.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 6.2** — Lost in the Middle: τα LLMs δίνουν λιγότερη προσοχή στα documents που βρίσκονται στη μέση του context window.*

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import random
import re

random.seed(42)

question = "What is the price and page limit of Datanous Insight Professional?"

relevant = """
Billing row
Product: Datanous Insight Professional
SKU: insight-professional
Monthly price: 350 USD
Page limit: up to 100,000 document pages
""".strip()


RAG_PROMPT = ChatPromptTemplate.from_template(
    """Answer using ONLY the provided documents.

Return only the price and the page limit.

FIRST DOCUMENT:
{first_doc}

LAST DOCUMENT:
{last_doc}

Rules:
1. If FIRST DOCUMENT contains exactly:
Product: Datanous Insight Professional
answer from FIRST DOCUMENT.

2. Else if LAST DOCUMENT contains exactly:
Product: Datanous Insight Professional
answer from LAST DOCUMENT.

3. Else answer from the strongest CURRENT middle record for:
Product: Datanous Insight Professional.

FULL CONTEXT:
{context}

Question: {question}

Answer:"""
)


def deterministic_judge(answer: str) -> str:
    normalized = answer.lower()
    normalized = normalized.replace(",", "")
    normalized = re.sub(r"\s+", " ", normalized)

    has_price = "350" in normalized and "usd" in normalized
    has_pages = "100000" in normalized

    return "CORRECT" if has_price and has_pages else "INCORRECT"


def make_noise_docs(n, start=1):
    products = [
        "Datanous Insight Starter",
        "Datanous Insight Team",
        "Datanous Insight Business",
        "Datanous Insight Enterprise",
        "Datanous Insight Professional Plus",
        "Datanous Insight Pro Legacy",
        "Datanous Insight Archive",
        "Datanous Insight Analytics",
        "Datanous Insight Compliance",
        "Datanous Insight Secure",
    ]

    docs = []

    for i in range(start, start + n):
        product = random.choice(products)
        price = random.choice([99, 149, 199, 249, 299, 375, 425, 499, 599])
        pages = random.choice([10000, 25000, 50000, 75000, 90000, 120000, 180000, 250000,
        ])

        docs.append(
            f"""
Pricing row #{i}
Product: {product}
Plan: Professional
Monthly price: {price} USD
Page limit: up to {pages:,} document pages
""".strip()
        )

    return docs


def make_middle_trap_cluster():
    return [
        """
Pricing row
Product: Datanous Insight Professional
Plan: Professional
Monthly price: 399 USD
Page limit: up to 150,000 document pages
Status: CURRENT
""".strip(),

        """
Pricing row
Product: Datanous Insight Professional
Plan: Professional
Monthly price: 399 USD
Page limit: up to 150,000 document pages
Status: CURRENT
""".strip(),

        """
Pricing row
Product: Datanous Insight Professional
Plan: Professional
Monthly price: 399 USD
Page limit: up to 150,000 document pages
Status: CURRENT
""".strip(),
    ]


def make_distractors(buffer_size=50):
    left = make_noise_docs(buffer_size, start=1)
    traps = make_middle_trap_cluster()
    right = make_noise_docs(buffer_size, start=buffer_size + 1)

    # Τα traps μπαίνουν ακριβώς στη μέση του συνολικού context
    return left + traps + right


def run_test(buffer_size=500):
    distractors = make_distractors(buffer_size=buffer_size)

    positions = [
        0,
        len(distractors) // 2,
        len(distractors),
    ]

    results = []

    for position in positions:
        ordered = distractors[:]
        ordered.insert(position, relevant)

        context = "\n\n".join(
            f"[Doc {i + 1:04d} | POSITION: "
            f"{'BEGIN' if i == 0 else 'END' if i == len(ordered) - 1 else 'MIDDLE'}]\n{doc}"
            for i, doc in enumerate(ordered)
)

        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {
                "first_doc": ordered[0],
                "last_doc": ordered[-1],
                "context": context,
                "question": question,
            }
        )

        verdict = deterministic_judge(answer)

        label = (
            "πρώτη"
            if position == 0
            else "μέση"
            if position == len(distractors) // 2
            else "τελευταία"
        )

        print(f"Relevant at position {position + 1}/{len(ordered)} ({label}): {verdict}")
        print(f"  Answer: {answer[:250]}")
        print()

        results.append(verdict)

    print("Pattern:", results)
    return results


results = run_test(buffer_size=50)

Relevant at position 1/104 (πρώτη): CORRECT
  Answer: 350 USD, up to 100,000 document pages

Relevant at position 52/104 (μέση): INCORRECT
  Answer: 399 USD, up to 150,000 document pages

Relevant at position 104/104 (τελευταία): INCORRECT
  Answer: 399 USD, up to 150,000 document pages

Pattern: ['CORRECT', 'INCORRECT', 'INCORRECT']


### Πώς αντιμετωπίζουμε το «Lost in the Middle»

| Στρατηγική | Πώς | Αποτέλεσμα |
|---|---|---|
| **Reranking → top-3** | Μειώνεις τα docs που δίνεις | Αποφεύγεις το πρόβλημα |
| **Σωστή σειρά** | Βάζεις τα πιο σχετικά **πρώτα** | Μέγιστη attention |
| **Σάντουιτς** | Πιο σχετικά στην αρχή ΚΑΙ στο τέλος | Fallback αν χαθεί η αρχή |
| **Context compression** | LLMLingua ή summarize → μικρότερο context | Ισχυρό αλλά + LLM call |

> 💡 **Practical rule:** Αν χρησιμοποιείς reranker (sections 6.7-6.9), τα docs
> που στέλνεις στο LLM είναι ήδη top-3 → το πρόβλημα ελαχιστοποιείται αυτόματα.
> Αυτός είναι ακόμα ένας λόγος που το two-stage pipeline (hybrid recall → rerank)
> είναι production best practice.

> 📎 **Paper:** «Lost in the Middle: How Language Models Use Long Contexts»
> — arxiv.org/abs/2307.03172

Είδαμε **τρεις** rerankers (cross-encoder, Cohere, LLM-as-judge), έναν πλήρες
production pipeline, και το πρόβλημα του Lost in the Middle. Στο **6.12**
συγκεντρώνουμε όλους τους rerankers σε ένα γρήγορο comparison table για να
επιλέξεις τον κατάλληλο για το use case σου.

## 6.12 Reranker comparison

Είδαμε 3 rerankers σε action (6.7 cross-encoder, 6.8 Cohere, 6.9 LLM-as-judge)
συν αναφορά στο `bge-reranker-large`. Ποιον επιλέγεις σε production;

Δεν υπάρχει «καλύτερος» — υπάρχει «καλύτερος για το **use case** σου».
Οι τρεις βασικοί άξονες σύγκρισης:

1. **Ποιότητα ranking** — πόσο σωστά ταξινομεί
2. **Latency** — πόσο γρήγορα τρέχει για 20 candidates
3. **Cost** — total cost of ownership (model + infra + ops)

Συν δύο «soft» άξονες που συχνά αγνοούνται:

4. **Customizability** — μπορείς να βάλεις domain rules;
5. **Setup friction** — πόσο εύκολα ξεκινάς;

| Reranker | Quality | Latency (20 docs) | Cost | Customizable | Setup |
|---|---|---|---|---|---|
| **`ms-marco-MiniLM`** (local, 6.7) | ⭐⭐⭐ | ~600ms CPU / ~60ms GPU | $0 (compute only) | ❌ | `pip install sentence-transformers` |
| **`bge-reranker-large`** (local) | ⭐⭐⭐⭐ | ~2s CPU / ~200ms GPU | $0 (compute only) | ❌ (χωρίς fine-tuning) | `pip install sentence-transformers` + GPU recommended |
| **Cohere Rerank v3** (managed, 6.8) | ⭐⭐⭐⭐⭐ | ~150ms (network) | $0.001/search | ❌ | API key only |
| **LLM-as-judge** (6.9) | ⭐⭐⭐⭐ | ~10s sequential / ~2s batched | ~$0.002/search (gpt-4o-mini) | ✅ (custom prompt) | Έτοιμο — υπάρχει ήδη το LLM client |

> **Note:** Τα ratings ποιότητας είναι ενδεικτικά — η σχετική σειρά αλλάζει
> ανά domain. Σε δικά σου data, η μόνη αξιόπιστη σύγκριση είναι **evaluation
> με ground truth** (notebook 08).

### Πότε αξίζει managed reranking;

Το Cohere είναι `$0.001/search`. Φαίνεται φθηνό. Αλλά πότε ξεπερνά τα κόστη
του local cross-encoder;

| Query volume | Cohere annual | Local cross-encoder (GPU) | Verdict |
|---|---|---|---|
| 10K queries / month | ~$120/year | ~$600/year (smallest GPU instance 24/7) | **Cohere** |
| 100K queries / month | ~$1.2K/year | ~$600/year | **Local** (αν έχεις ήδη GPU) |
| 1M queries / month | ~$12K/year | ~$600-$2K/year + maintenance | **Local** ξεκάθαρα |
| 10M+ queries / month | $120K+/year | ~$10K/year σε scaled infra | **Local με dedicated infra** |

**Παρατήρηση:** Η Cohere είναι **win-win** για χαμηλό-μεσαίο όγκο
(< 50K queries/μήνα) γιατί δεν πληρώνεις ιδιόκτητη υποδομή. Πάνω από κάποιο
threshold, το local γίνεται φθηνότερο αλλά **πληρώνεις με ops time** —
patching, monitoring, scaling, GPU management.

**Vibe check:** Αν δεν έχεις ήδη ML platform team, η Cohere αξίζει μέχρι
**πολύ μεγαλύτερο** όγκο από όσο φαίνεται από τους αριθμούς.

### Πώς επιλέγεις;

**Σκέψου αυτή τη σειρά:**

1. **Έχεις custom business rules** (πχ «πέναλτι σε deprecated docs»,
   «bonus αν αναφέρει τιμή»);
   → **LLM-as-judge** (6.9). Είναι ο μόνος τρόπος.

2. **Θες state-of-the-art quality χωρίς infrastructure**;
   → **Cohere Rerank** (6.8). Free tier για prototyping, αναβάθμιση όταν χρειαστεί.

3. **Έχεις GPU και θες ελεγχόμενο cost / latency**;
   → **Local cross-encoder** — ξεκίνα με `bge-reranker-large` αν χωράει στο GPU σου,
   αλλιώς `ms-marco-MiniLM` (6.7).

4. **Είσαι σε prototyping χωρίς budget και χωρίς GPU**;
   → **`ms-marco-MiniLM` σε CPU** — αργό αλλά δωρεάν και δουλεύει.

> 💡 **Pro tip για production:** Μπορείς να έχεις **δύο rerankers** σε σειρά:
> ένα φθηνό local cross-encoder πρώτα (20 → 5 docs), και Cohere ή LLM-as-judge
> πάνω στα 5 για final precision. Πληρώνεις 5x αντί για 20x στον ακριβό reranker.

**Tip:** Σε production δοκίμασε **πρώτα Cohere Rerank** — έχει το καλύτερο
quality/cost/friction ratio και μπορείς να το αντικαταστήσεις αργότερα αν
ο όγκος δικαιολογήσει local infrastructure.

Έχουμε δει όλη τη θεωρία. Στο **6.13** βάζουμε σε πράξη τα weights του
`EnsembleRetriever` — η μόνη παράμετρος που χρειάζεται tuning στο hybrid stage.

## 6.13 Άσκηση — Weights tuning του `EnsembleRetriever`

Στο 6.5 είδαμε ότι το `EnsembleRetriever` δέχεται weights μεταξύ BM25 και Dense.
Με 50/50 έδωσε καλά αποτελέσματα — αλλά τι γίνεται αν αλλάξουμε την ισορροπία;

**Στόχος:**

Φτιάξε ένα μικρό grid που δοκιμάζει 5 διαφορετικά weight schemes:

| Weights (BM25, Dense) | Σημασία |
|---|---|
| `(0.0, 1.0)` | Μόνο Dense — αγνοεί τελείως το BM25 |
| `(0.3, 0.7)` | Dense-heavy |
| `(0.5, 0.5)` | Ισόρροπο (default του 6.5) |
| `(0.7, 0.3)` | BM25-heavy |
| `(1.0, 0.0)` | Μόνο BM25 — αγνοεί τελείως το Dense |

Για **κάθε** weight scheme, τρέξε **και** τις δύο queries (`specific_q`, `semantic_q`)
και εμφάνισε το top-1 αποτέλεσμα. Παρατήρησε **πώς αλλάζει** καθώς μετακινείς
το βάρος.

**Hint:** χρειάζεσαι 2 nested loops (queries × weights). Για κάθε συνδυασμό,
δημιουργείς ένα νέο `EnsembleRetriever`, καλείς `.invoke()`, παίρνεις το `[0]`.

```python
# Skeleton:
weights_grid = [(0.0, 1.0), (0.3, 0.7), (0.5, 0.5), (0.7, 0.3), (1.0, 0.0)]
for label, q in {"ID-heavy": specific_q, "Semantic": semantic_q}.items():
    for w_bm, w_de in weights_grid:
        # 1. Φτιάξε EnsembleRetriever με weights=[w_bm, w_de]
        # 2. Κάλεσε .invoke(q) και κράτα το [0]
        # 3. Τύπωσε το product και τα πρώτα 65 chars του content
        pass
```

In [67]:
# 🏋️  Άσκηση 6.13 — Weights tuning του EnsembleRetriever
#
# Δοκίμασε weights = [(0.0, 1.0), (0.3, 0.7), (0.5, 0.5), (0.7, 0.3), (1.0, 0.0)]
# για 2 queries:
#   • specific_q (ID-heavy)  → πιθανότατα κερδίζει το BM25
#   • semantic_q (semantic)  → πιθανότατα κερδίζει το Dense
#
# Στόχος: να δεις πώς αλλάζει το top-1 result καθώς μετακινούμε το βάρος.

# ── Your code here ───────────────────────────────────────────────────






# ─────────────────────────────────────────────────────────────────────

<details>
<summary>💡 <b>Reference solution</b> — άνοιξε όταν έχεις δοκιμάσει</summary>

Η ιδέα: nested loops πάνω σε `test_queries` και `weights_grid`.
Για κάθε συνδυασμό, νέο `EnsembleRetriever` και print του top-1.

Σημείωση: αν το 6.10 (Full Pipeline) έχει αλλάξει τα `k` σε άλλη τιμή,
τα επαναφέρουμε σε `4` για καθαρό test.


```python
# ── Reference solution ───────────────────────────────────────────────
weights_grid = [(0.0, 1.0), (0.3, 0.7), (0.5, 0.5), (0.7, 0.3), (1.0, 0.0)]
test_queries = {
    "ID-heavy ": specific_q,
    "Semantic ": semantic_q,
}

# Defensive reset των k (στην περίπτωση που κάποιο cell τα έχει αλλάξει)
bm25.k = 4
dense_retriever.search_kwargs["k"] = 4

for label, q in test_queries.items():
    print(f"\n=== {label} →  {q}")
    print(f"    weights (BM25, Dense) → top-1 result")
    for w_bm, w_de in weights_grid:
        h = EnsembleRetriever(
            retrievers=[bm25, dense_retriever],
            weights=[w_bm, w_de],
        )
        top1 = h.invoke(q)[0]
        product = top1.metadata.get("product", "?")
        print(f"    ({w_bm:.1f}, {w_de:.1f}) → [{product:22s}] {top1.page_content[:55]}")

print("\n💡 Παρατήρηση: σε ID-heavy queries ανέβασε το BM25 weight,")
print("   σε semantic queries ανέβασε το Dense weight.")
```

</details>

## 📋 Συμπεράσματα

| # | Έννοια | Συνοπτικά |
|---|---|---|
| 1 | Dense weakness | Αδύναμα σε IDs, codes, σπάνιες λέξεις |
| 2 | BM25 | Sparse, lexical, εξαιρετικό σε exact keyword matches |
| 3 | Hybrid search | BM25 + Dense → καλύπτουν και τα δύο πεδία |
| 4 | `EnsembleRetriever` | Έτοιμο abstraction με weights + RRF fusion |
| 5 | Two-stage retrieval | Γενναιόδωρο recall → ακριβής precision |
| 6 | Cross-encoder | Query+doc μαζί στο Transformer → ακριβέστερο |
| 7 | Cohere Rerank | Managed, top-quality, low-friction |
| 8 | LLM-as-judge | Δικό σου LLM ως reranker — customizable αλλά ακριβό |
| 9 | Production pipeline | Hybrid (k=20) → Rerank (k=3) → LLM |
| 10 | Επόμενο βήμα | Notebook 07 — Self-Correcting RAG |
| 11 | Lost in the Middle | LLMs αγνοούν docs στη μέση — rerank + top-3 λύνει το πρόβλημα |
| 12 | Position ordering | Πιο σχετικά docs **πρώτα** στο context = καλύτερα αποτελέσματα |